# check2HGI MTL — Colab Training Template

Self-contained notebook for running the **north-star B3 config** on Colab T4.
Mirrors the canonical CLI from `docs/studies/check2hgi/NORTH_STAR.md`.

**See `docs/COLAB_GUIDE.md` for the operational manual** — every cell
below is documented there with the *why* (memory pitfalls, perf rationale,
verification recipes).

**Workflow:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run cells ①–④ once per session
3. Edit cell ⑤ for your run
4. Run ⑥ (detached launch) → ⑦ (monitor) → ⑧ (results)

---
## ① Mount Drive + paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive/mestrado/PoiMtlNet')
DATA_ROOT    = DRIVE_ROOT / 'data'
OUTPUT_DIR   = DRIVE_ROOT / 'output'
RESULTS_ROOT = DRIVE_ROOT / 'results'

for p in [DATA_ROOT, OUTPUT_DIR, RESULTS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

os.environ['DATA_ROOT']    = str(DATA_ROOT)
os.environ['OUTPUT_DIR']   = str(OUTPUT_DIR)
os.environ['RESULTS_ROOT'] = str(RESULTS_ROOT)

for label, p in [('DATA_ROOT', DATA_ROOT), ('OUTPUT_DIR', OUTPUT_DIR), ('RESULTS_ROOT', RESULTS_ROOT)]:
    print(f'{label:13s} {p}  (exists={p.exists()})')

---
## ② Clone repo + checkout the perf branch

`feat/colab-gpu-perf` is branched off `worktree-check2hgi-mtl` and contains
the CUDA/T4-specific perf patches (num_workers=0, chunked-MRR,
cudnn.benchmark, vectorised OOD mask, rank cache, between-fold cache clear).
All changes are quality-neutral and MPS-safe.

In [ ]:
REPO_DIR = Path('/content/PoiMtlNet')
BRANCH = 'feat/colab-gpu-perf'   # change to worktree-check2hgi-mtl if you want the no-perf base

if not REPO_DIR.exists():
    !git clone https://github.com/VitorHugoOli/PoiMtlNet.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch --quiet origin {BRANCH}
!git checkout {BRANCH}
!git pull --ff-only
!git log --oneline -3

# Sanity: the check2HGI-specific CLI flags must be present.
import subprocess
for flag in ('task-a-input-type', 'task-set', 'reg-head', 'cat-head'):
    n = int(subprocess.run(['grep', '-c', flag, 'scripts/train.py'],
                            capture_output=True, text=True).stdout.strip() or 0)
    print(f"  {'OK' if n > 0 else 'MISSING'} --{flag}")

---
## ③ Install deps + confirm CUDA

In [ ]:
!pip install -q -r requirements_colab.txt

import torch
_tv = torch.__version__
_pyg_url = f'https://data.pyg.org/whl/torch-{_tv}.html'
!pip uninstall -q -y torch-scatter torch-sparse torch-cluster torch-geometric 2>/dev/null || true
!pip install -q torch-scatter -f {_pyg_url}
!pip install -q torch-sparse  -f {_pyg_url}
!pip install -q torch-cluster -f {_pyg_url}
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

import sys
for sub in ('src', 'research'):
    p = str(REPO_DIR / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

from configs.globals import DEVICE
assert DEVICE.type == 'cuda', f'Expected CUDA, got {DEVICE} — switch Runtime to T4 GPU and retry.'
print(f'Device: {DEVICE} (cudnn.benchmark = {torch.backends.cudnn.benchmark})')

---
## ④ Configure the run

Edit `STATE` for your target. The other fields encode the post-F27 north-star
B3 config and should not be changed for paper-table runs.

In [ ]:
# ── EDIT HERE ────────────────────────────────────────────────────────────────
STATE  = 'florida'      # 'alabama' | 'arizona' | 'florida'
FOLDS  = 5
EPOCHS = 50
SEED   = 42
TAG    = ''             # optional suffix appended to the log filename
# ─────────────────────────────────────────────────────────────────────────────

TRANSITION_PATH = OUTPUT_DIR / 'check2hgi' / STATE / 'region_transition_log.pt'
LOG_FILE        = RESULTS_ROOT / f'{STATE}_{FOLDS}f_{EPOCHS}ep_northstar{TAG}.log'

checks = {
    'check-in embeddings': OUTPUT_DIR / 'check2hgi' / STATE / 'embeddings.parquet',
    'next.parquet':        OUTPUT_DIR / 'check2hgi' / STATE / 'input' / 'next.parquet',
    'transition_log':      TRANSITION_PATH,
}
for name, p in checks.items():
    print(f"  {'OK' if p.exists() else 'MISSING'}  {name}: {p}")

_cmd = (
    f'python -u scripts/train.py'
    f' --state {STATE}'
    f' --task mtl'
    f' --task-set check2hgi_next_region'
    f' --engine check2hgi'
    f' --folds {FOLDS}'
    f' --epochs {EPOCHS}'
    f' --seed {SEED}'
    f' --task-a-input-type checkin'
    f' --task-b-input-type region'
    f' --model mtlnet_crossattn'
    f' --mtl-loss static_weight'
    f' --category-weight 0.75'
    f' --cat-head next_gru'
    f' --reg-head next_getnext_hard'
    f' --reg-head-param d_model=256'
    f' --reg-head-param num_heads=8'
    f' --reg-head-param transition_path={TRANSITION_PATH}'
    f' --max-lr 0.003'
    f' --gradient-accumulation-steps 1'
    f' --no-checkpoints'
)
print(f'\nCommand:\n{_cmd}\n\nLog: {LOG_FILE}')

---
## ⑤ Launch as a detached subprocess

`nohup` + `start_new_session=True` ensures the run survives:
- MCP/cell timeouts (long `!{cmd}` cells get SIGINT'd around 5 min)
- Notebook tab close, kernel disconnect, browser refresh

Output goes to a Drive log file via direct stdout redirect.
**Do not use `!{cmd}` for runs > 5 min.**

In [ ]:
import subprocess, shlex

# Defensive: kill any leftover trainer from a prior cell run.
subprocess.run(['pkill', '-9', '-f', 'scripts/train.py'], capture_output=True)

with LOG_FILE.open('w') as lf:
    proc = subprocess.Popen(
        ['nohup'] + shlex.split(_cmd),
        stdout=lf, stderr=subprocess.STDOUT,
        cwd='/content/PoiMtlNet',
        start_new_session=True,
    )

print(f'Training launched — PID {proc.pid} (detached)')
print(f'Branch: {BRANCH}')
print(f'Log: {LOG_FILE}')
print(f'\nMonitor with cell ⑥. The process keeps running even if this notebook disconnects.')

---
## ⑥ Monitor (re-run anytime to refresh)

In [ ]:
import re, subprocess

text = LOG_FILE.read_text() if LOG_FILE.exists() else ''
lines = text.splitlines()

# Process status
r = subprocess.run(['pgrep', '-f', f'state {STATE}'], capture_output=True, text=True)
pid = r.stdout.strip()
if pid:
    info = subprocess.run(['ps', '-p', pid, '-o', 'pid,etime,rss,stat', '--no-headers'],
                          capture_output=True, text=True).stdout.strip()
    print(f'Process: {info}')
else:
    print('Process: GONE (run finished or crashed — check log tail)')

# Fold + last progress
fold_markers = [l for l in lines if 'FOLD' in l and '====' in l]
n_started = len(fold_markers)
if fold_markers:
    print('\n' + fold_markers[-1].split(' - INFO  - ')[-1][:80])
last = next((l for l in reversed(lines) if 'best=' in l and 'Validating' not in l), None)
if last:
    print(last[:200])

# ETA — tqdm batches are PER FOLD (50 epochs combined)
m = re.search(r'(\d+)/(\d+)\s*\[(\d+):(\d+)<(\d+):(\d+),\s*(\d+\.\d+)batch/s', last or '')
if m:
    bnow, btotal = int(m.group(1)), int(m.group(2))
    elapsed_s   = int(m.group(3)) * 60 + int(m.group(4))
    remaining_s = int(m.group(5)) * 60 + int(m.group(6))
    rate        = float(m.group(7))
    fold_eta_min = remaining_s / 60
    avg_fold_min = (elapsed_s + remaining_s) / 60
    folds_remaining = FOLDS - n_started + 1
    total_eta_min = fold_eta_min + (folds_remaining - 1) * avg_fold_min
    print(f'\nThis fold: {bnow}/{btotal} batches @ {rate:.2f} batch/s — '
          f'{fold_eta_min:.1f} min remaining (fold takes ~{avg_fold_min:.1f} min)')
    print(f'Folds done: {n_started - 1}/{FOLDS} — full-run ETA: ~{total_eta_min:.1f} min')

# Completed fold summaries
summaries = [l for l in lines if ('next_category' in l or 'next_region' in l) and '|' in l and '%' in l]
if summaries:
    print('\nCompleted fold summaries:')
    for l in summaries[-12:]:
        print('  ' + l.split(' - INFO  - ')[-1][:120])

# GPU + RAM
subprocess.run(['nvidia-smi', '--query-gpu=memory.used,memory.free,utilization.gpu',
                '--format=csv,noheader'])

---
## ⑦ Inspect aggregated results

In [ ]:
import json

results_dir = RESULTS_ROOT / 'check2hgi' / STATE
runs = sorted(results_dir.glob('mtlnet_*'), key=lambda p: p.stat().st_mtime, reverse=True)
if not runs:
    print('No runs found yet.')
else:
    latest = runs[0]
    summary = latest / 'summary' / 'full_summary.json'
    print(f'Run dir: {latest.name}\n')
    if summary.exists():
        data = json.load(summary.open())
        for task in ('next_category', 'next_region', 'model'):
            if task in data:
                print(f'{task}:')
                for k, stats in data[task].items():
                    if isinstance(stats, dict) and 'mean' in stats:
                        print(f'  {k:25s} {stats["mean"]:.4f} ± {stats["std"]:.4f}')
                print()
    else:
        print('summary/full_summary.json not yet written — run still in progress.')

---
## ⑧ (Optional) Verify the running process loaded fresh code

Use this if you `git pull`'d new commits during a run and need to confirm
the subprocess actually loaded them. Subprocesses don't share imports with
the notebook kernel, so a `pull → relaunch` cycle picks up new code without
needing a kernel restart — but verify if in doubt.

In [ ]:
import subprocess
from datetime import datetime

# Replace with the PID printed by cell ⑤
PID = proc.pid if 'proc' in dir() else None

if PID:
    print(f'PID {PID}:')
    r = subprocess.run(['ps', '-p', str(PID), '-o', 'lstart='],
                       capture_output=True, text=True)
    print(f'  started: {r.stdout.strip()}')

for path in ['src/tracking/metrics.py', 'src/training/runners/mtl_eval.py', 'src/utils/mps.py']:
    p = Path(f'/content/PoiMtlNet/{path}')
    print(f'  {path:38s}  src mtime: {datetime.fromtimestamp(p.stat().st_mtime)}')

for pyc_dir in ['src/tracking/__pycache__', 'src/training/runners/__pycache__', 'src/utils/__pycache__']:
    base = Path(f'/content/PoiMtlNet/{pyc_dir}')
    if base.exists():
        for pyc in base.glob('*.cpython-*.pyc'):
            if pyc.stem.split('.')[0] in ('metrics', 'mtl_eval', 'mps'):
                mt = datetime.fromtimestamp(pyc.stat().st_mtime)
                print(f'  {pyc.relative_to("/content/PoiMtlNet")} compiled: {mt}')

print('\nDecisive: .pyc compile time must be AFTER src mtime AND AFTER subprocess start.')
print('If not, the running process is on stale code — kill and relaunch.')